# Historical inference notebook

I kept this notebook as a record of our inference experiments. It requires the original Kaggle inputs, external model code, weights and GPU environment. The supported local example is the [CPU walkthrough](../../notebooks/01_workflow_walkthrough.ipynb).

Outputs and execution counts are cleared in this copy. Original file hashes and saved-run details are recorded in the [archive manifest](../source_manifest.json) and [historical notes](../../docs/HISTORICAL_NOTES.md).

The original whole-workspace deletion call is replaced with an explicit stop. The separate concatenation error is documented in the historical notes.


In [ ]:
import os
MODEL_TYPE='protenix'
VALIDATION=False
SEED=42
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
TRRNA_THRESHOLD=300
NUM_CPU=os.cpu_count()
SEED=42
trRNA_input_ids = []
results = {}

## Install requirements 

In [ ]:
# if MODEL_TYPE=='protenix' and VALIDATION:
#     !pip install --no-deps protenix
#     !pip install biopython
#     !pip install ml-collections
#     !pip install biotite==1.0.1
#     !pip install rdkit
# !export PROTENIX_DATA_ROOT_DIR=/kaggle/input/protenix-checkpoints

In [ ]:
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/protenix-0.4.6-py3-none-any.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/rdkit-2024.9.6-cp310-cp310-manylinux_2_28_x86_64.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/ml_collections-1.1.0-py3-none-any.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/biopython-1.85-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/pyrosetta-2025.13-cp310-cp310-linux_x86_64.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/blosc-1.11.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/ml_collections-1.1.0-py3-none-any.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/biotraj-1.2.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/biotite-1.0.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/biopandas-0.5.1-py3-none-any.whl'
!pip install --no-deps '/kaggle/input/dependencies-tr-pr/looseversion-1.1.2-py3-none-any.whl'

In [ ]:
! mkdir /af3-dev 
! ln -s /kaggle/input/protenix-checkpoints /af3-dev/release_data
! ls /af3-dev/release_data/

## Helper scripts

In [ ]:
import Bio

from copy import deepcopy

import pandas as pd
from Bio.PDB import Atom, Model, Chain, Residue, Structure, PDBParser
from Bio import SeqIO
import os, sys
import re
import numpy as np
import torch

import matplotlib
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
time0=time.time()

print('IMPORT OK !!!!')

In [ ]:
!cd /kaggle/input/data-for-demo-for-rhofold-plus-with-kaggle-msa/RhoFold-main
!dir

In [ ]:
PYTHON = sys.executable
print('PYTHON',PYTHON)

RHONET_DIR=\
'/kaggle/input/data-for-demo-for-rhofold-plus-with-kaggle-msa/RhoFold-main'
#'<your downloaded rhofold repo>/RhoFold-main'

USALIGN = \
'/kaggle/working//USalign'
#'<your us align path>/USalign'

os.system('cp /kaggle/input/usalign/USalign /kaggle/working/')
os.system('sudo chmod u+x /kaggle/working//USalign')
sys.path.append(RHONET_DIR)


DATA_KAGGLE_DIR = '/kaggle/input/stanford-rna-3d-folding'


# helper ----
class dotdict(dict):
	__setattr__ = dict.__setitem__
	__delattr__ = dict.__delitem__

	def __getattr__(self, name):
		try:
			return self[name]
		except KeyError:
			raise AttributeError(name)

# visualisation helper ----
def set_aspect_equal(ax):
	x_limits = ax.get_xlim()
	y_limits = ax.get_ylim()
	z_limits = ax.get_zlim()

	# Compute the mean of each axis
	x_middle = np.mean(x_limits)
	y_middle = np.mean(y_limits)
	z_middle = np.mean(z_limits)

	# Compute the max range across all axes
	max_range = max(x_limits[1] - x_limits[0],
					y_limits[1] - y_limits[0],
					z_limits[1] - z_limits[0]) / 2.0

	# Set the new limits to ensure equal scaling
	ax.set_xlim(x_middle - max_range, x_middle + max_range)
	ax.set_ylim(y_middle - max_range, y_middle + max_range)
	ax.set_zlim(z_middle - max_range, z_middle + max_range)




# xyz df helper --------------------
def get_truth_df(target_id):
    truth_df = LABEL_DF[LABEL_DF['target_id'] == target_id]
    truth_df = truth_df.reset_index(drop=True)
    return truth_df

def parse_output_to_df(output, seq, target_id):
    df = []
    chain_data = []
    for i, res in enumerate(seq):
        d=dict(ID = target_id,
                    resname=res,
                    resid=i+1)
        for n in range(len(output)):
            d={**d, f'x_{n+1}': round(output[n,i,0].item(),3),
                     f'y_{n+1}': round(output[n,i,1].item(),3),
                     f'z_{n+1}': round(output[n,i,2].item(),3)}
        chain_data.append(d)

    if len(chain_data)!=0:
        chain_df = pd.DataFrame(chain_data)
        df.append(chain_df)
        ##print(chain_df)
    return df

def parse_pdb_to_df(pdb_file, target_id):
    parser = PDBParser()
    structure = parser.get_structure('', pdb_file)

    df = []
    for model in structure:
        for chain in model:
            print(chain)
            chain_data = []
            for residue in chain:
                # print(residue)
                if residue.get_resname() in ['A', 'U', 'G', 'C']:
                    # Check if the residue has a C1' atom
                    if 'C1\'' in residue:
                        atom = residue['C1\'']
                        xyz = atom.get_coord()
                        resname = residue.get_resname()
                        resid = residue.get_id()[1]

                        #todo detect discontinous: resid = prev_resid+1
                        #ID	resname	resid	x_1	y_1	z_1
                        chain_data.append(dict(
                            ID = target_id+'_'+str(resid),
                            resname=resname,
                            resid=resid,
                            x_1=xyz[0],
                            y_1=xyz[1],
                            z_1=xyz[2],
                        ))
                        ##print(f"Residue {resname} {resid}, Atom: {atom.get_name()}, xyz: {xyz}")

            if len(chain_data)!=0:
                chain_df = pd.DataFrame(chain_data)
                df.append(chain_df)
                ##print(chain_df)
    return df

# usalign helper --------------------
def write_target_line(
    atom_name, atom_serial, residue_name, chain_id, residue_num, x_coord, y_coord, z_coord, occupancy=1.0, b_factor=0.0, atom_type='P'
):
    """
    Writes a single line of PDB format based on provided atom information.

    Args:
        atom_name (str): Name of the atom (e.g., "N", "CA").
        atom_serial (int): Atom serial number.
        residue_name (str): Residue name (e.g., "ALA").
        chain_id (str): Chain identifier.
        residue_num (int): Residue number.
        x_coord (float): X coordinate.
        y_coord (float): Y coordinate.
        z_coord (float): Z coordinate.
        occupancy (float, optional): Occupancy value (default: 1.0).
        b_factor (float, optional): B-factor value (default: 0.0).

    Returns:
        str: A single line of PDB string.
    """
    return f'ATOM  {atom_serial:>5d}  {atom_name:<5s} {residue_name:<3s} {residue_num:>3d}    {x_coord:>8.3f}{y_coord:>8.3f}{z_coord:>8.3f}{occupancy:>6.2f}{b_factor:>6.2f}           {atom_type}\n'

def write_xyz_to_pdb(df, pdb_file, xyz_id = 1):
    resolved_cnt = 0
    with open(pdb_file, 'w') as target_file:
        for _, row in df.iterrows():
            x_coord = row[f'x_{xyz_id}']
            y_coord = row[f'y_{xyz_id}']
            z_coord = row[f'z_{xyz_id}']

            if x_coord > -1e17 and y_coord > -1e17 and z_coord > -1e17:
                resolved_cnt += 1
                target_line = write_target_line(
                    atom_name="C1'",
                    atom_serial=int(row['resid']),
                    residue_name=row['resname'],
                    chain_id='0',
                    residue_num=int(row['resid']),
                    x_coord=x_coord,
                    y_coord=y_coord,
                    z_coord=z_coord,
                    atom_type='C',
                )
                target_file.write(target_line)
    return resolved_cnt

def parse_usalign_for_tm_score(output):
    # Extract TM-score based on length of reference structure (second)
    tm_score_match = re.findall(r'TM-score=\s+([\d.]+)', output)[1]
    if not tm_score_match:
        raise ValueError('No TM score found')
    return float(tm_score_match)

def parse_usalign_for_transform(output):
    # Locate the rotation matrix section
    matrix_lines = []
    found_matrix = False

    for line in output.splitlines():
        if "The rotation matrix to rotate Structure_1 to Structure_2" in line:
            found_matrix = True
        elif found_matrix and re.match(r'^\d+\s+[-\d.]+\s+[-\d.]+\s+[-\d.]+\s+[-\d.]+$', line):
            matrix_lines.append(line)
        elif found_matrix and not line.strip():
            break  # Stop parsing if an empty line is encountered after the matrix

    # Parse the rotation matrix values
    rotation_matrix = []
    for line in matrix_lines:
        parts = line.split()
        row_values = list(map(float, parts[1:]))  # Skip the first column (index)
        rotation_matrix.append(row_values)

    return np.array(rotation_matrix)

def call_usalign(predict_df, truth_df, verbose=1):
    truth_pdb = '~truth.pdb'
    predict_pdb = '~predict.pdb'
    write_xyz_to_pdb(predict_df, predict_pdb, xyz_id=1)
    write_xyz_to_pdb(truth_df, truth_pdb, xyz_id=1)

    command = f'{USALIGN} {predict_pdb} {truth_pdb} -atom " C1\'" -m -'
    output = os.popen(command).read()
    if verbose==1:
        print(output)
    tm_score = parse_usalign_for_tm_score(output)
    transform = parse_usalign_for_transform(output)
    return tm_score, transform

print('HELPER OK!!!')

In [ ]:
# PyRosetta setup for scoring Protenix predictions
import pyrosetta
from pyrosetta import pose_from_pdb, create_score_function
from pyrosetta.rosetta import core
import pandas as pd

# Initialize PyRosetta
pyrosetta.init(options='-mute all')

# Define the scoring function with RNA-specific weights
op_score = create_score_function('ref2015')
op_score.set_weight(core.scoring.atom_pair_constraint, 9.0)
op_score.set_weight(core.scoring.dihedral_constraint, 4.0)
op_score.set_weight(core.scoring.angle_constraint, 4.0)
op_score.set_weight(core.scoring.fa_rep, 9.0)
op_score.set_weight(core.scoring.rna_sugar_close, 9.0)
op_score.set_weight(core.scoring.fa_intra_rep, 9.0)
op_score.set_weight(core.scoring.rna_base_pair, 9.0)
op_score.set_weight(core.scoring.rna_base_stack, 9.0)

# Function to convert Protenix predictions to PDB and DataFrame
def protenix_to_pdb_and_df(coordinate, atom_array, pdb_path):
    # Create PDB lines
    lines = []
    for i, atom in enumerate(atom_array):
        # Extract atom information
        atom_name = atom.atom_name
        res_name = atom.res_name
        chain_id = atom.chain_id
        res_id = atom.res_id
        x, y, z = coordinate[i][0].item(), coordinate[i][1].item(), coordinate[i][2].item()
        # Create PDB line
        line = f"ATOM  {i+1:>5} {atom_name:<4} {res_name:<3} {chain_id:<1}{res_id:>4}    {x:>8.3f}{y:>8.3f}{z:>8.3f} 1.00  1.00           {atom.element:<2}\n"
        lines.append(line)
    # Write to PDB file
    with open(pdb_path, 'w') as f:
        f.writelines(lines)
    # Create DataFrame
    df = pd.DataFrame({
        'atom_name': [atom.atom_name for atom in atom_array],
        'res_name': [atom.res_name for atom in atom_array],
        'chain_id': [atom.chain_id for atom in atom_array],
        'res_id': [atom.res_id for atom in atom_array],
        'x': [coordinate[i][0].item() for i in range(len(atom_array))],
        'y': [coordinate[i][1].item() for i in range(len(atom_array))],
        'z': [coordinate[i][2].item() for i in range(len(atom_array))],
        'element': [atom.element for atom in atom_array]
    })
    return df

# Function to score a PDB file using op_score
def score_pdb(pdb_path):
    pose = pose_from_pdb(pdb_path)
    score = op_score(pose)
    return score
    

In [ ]:
import random
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = True
    torch.use_deterministic_algorithms(True)

In [ ]:
if MODEL_TYPE=='protenix':
    
    
    from runner.batch_inference import get_default_runner
    from runner.inference import update_inference_configs, InferenceRunner

    from protenix.data.infer_data_pipeline import InferenceDataset

    seed_everything(SEED)

    class DictDataset(InferenceDataset):
        def __init__(
            self,
            seq_list: list,
            dump_dir: str,
            id_list: list = None,
            use_msa: bool = False,
        ) -> None:

            self.dump_dir = dump_dir
            self.use_msa = use_msa
            if isinstance(id_list,type(None)):
                self.inputs = [{"sequences": 
                                [{"rnaSequence": 
                                  {"sequence": seq, 
                                   "count": 1}}],
                                "name": "query"} for seq in seq_list]
            else:
                self.inputs = [{"sequences": 
                                [{"rnaSequence": 
                                  {"sequence": seq, 
                                   "count": 1}}],
                                "name": i} for i, seq in zip(id_list,seq_list)]

In [ ]:
if MODEL_TYPE=='protenix':

    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from protenix.config.config import parse_configs

    configs_base["use_deepspeed_evo_attention"] = (
    os.environ.get("USE_DEEPSPEED_EVO_ATTENTION", False) == "true")
    configs_base["model"]["N_cycle"] = 10 #10
    configs_base["sample_diffusion"]["N_sample"] = (1 if VALIDATION else 5)
    configs_base["sample_diffusion"]["N_step"] = 200
    inference_configs['load_checkpoint_path']='/kaggle/input/protenix-checkpoints/model_v0.2.0.pt'
    configs = {**configs_base, **{"data": data_configs}, **inference_configs}

    configs = parse_configs(
            configs=configs,
            fill_required_with_null=True,
        )
    
    runner=InferenceRunner(configs)


In [ ]:
if VALIDATION:
    LABEL_DF = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/train_labels.csv')
    LABEL_DF['target_id'] = LABEL_DF['ID'].apply(lambda x: '_'.join(x.split('_')[:-1]))
    train_df=pd.read_csv('/kaggle/input/stanford-rna-3d-folding/train_sequences.csv')


In [ ]:

if MODEL_TYPE=='protenix' and VALIDATION:
    import warnings
    warnings.filterwarnings("ignore")  
    
    train_df['protenix_tm_score']=None
    dataset = DictDataset(train_df.sequence, dump_dir='output', id_list=train_df.target_id, use_msa=False)
    num_data = len(dataset)
    for i, seq in tqdm(enumerate(train_df.sequence),total=num_data):
        if train_df.loc[i,'protenix_tm_score']!=None:
            continue
        if len(seq)>300:
            continue
        target_id = train_df.loc[i,'target_id']
        truth_df = get_truth_df(target_id)
        if sum(~np.isnan(truth_df.x_1))<3:
            continue
        data, atom_array, data_error_message=dataset[i]
        if data_error_message!='':
            continue
        new_configs = update_inference_configs(configs, data["N_token"].item())
        runner.update_model_configs(new_configs)
        prediction = runner.predict(data)
        prediction=prediction['coordinate'][:,data['input_feature_dict']['atom_to_tokatom_idx']==12]       
        result = parse_output_to_df(prediction[:1], seq, target_id)[0]
        try:
            tm_score, transform = call_usalign(result, truth_df, verbose=0)
            train_df.loc[i,'protenix_tm_score']=tm_score
        except:
            pass
        if (time.time()-time0)>(12*3600-360):
            break
    train_df.to_csv('tm_scores.csv', index=False)
    print(train_df.protenix_tm_score.mean())
    display(train_df.protenix_tm_score.hist())

In [ ]:
if MODEL_TYPE=='protenix' and not VALIDATION:
    test_df=pd.read_csv('/kaggle/input/stanford-rna-3d-folding/test_sequences.csv')
    import warnings
    warnings.filterwarnings("ignore")  
    
    sequences = test_df.sequence.tolist()
    MAX_LEN = 960
    test_df['sequence'] = test_df['sequence'].apply(lambda x: x[:MAX_LEN] if len(x) > MAX_LEN else x)
    
    dataset = DictDataset(test_df.sequence, dump_dir='output', id_list=test_df.target_id, use_msa=False)
    num_data = len(dataset)
    for i, seq in tqdm(enumerate(test_df.sequence),total=num_data):
        original_seq = sequences[i]
        truncated = len(original_seq) > MAX_LEN
        
        try:
            data, atom_array, data_error_message=dataset[i]
            target_id = data["sample_name"]
            assert target_id==test_df.target_id[i]
            assert data_error_message==''
            
            new_configs = update_inference_configs(configs, data["N_token"].item())
            runner.update_model_configs(new_configs)
            prediction = runner.predict(data)

            # Score each of the 5 Protenix models using full coordinates
            scores = []
            protenix_dfs = []
            for model_idx in range(len(prediction['coordinate'])):
                pdb_path = f'protenix_model_{target_id}_{model_idx+1}.pdb'
                df = protenix_to_pdb_and_df(prediction['coordinate'][model_idx], atom_array, pdb_path)
                score = score_pdb(pdb_path)
                scores.append((model_idx, score))
                protenix_dfs.append(df)
            
            # Sort models by score (ascending, lower is better for energy scores)
            scores.sort(key=lambda x: x[1])
            sorted_indices = [idx for idx, _ in scores]
            
            # Reorder predictions based on scores (best to worst)
            sorted_prediction = prediction['coordinate'][sorted_indices]
            
            # Now extract C1' coordinates after sorting
            sorted_prediction = sorted_prediction[:, data['input_feature_dict']['atom_to_tokatom_idx']==12]
            
            # Convert the sorted predictions to DataFrame
            result = parse_output_to_df(sorted_prediction, seq, target_id)[0]
            
            # Store the best score for later comparison
            results[target_id] = {'df': result, 'best_score': scores[0][1]}
            
        except:
            target_id=test_df.target_id[i]
            print('Failed to predict', target_id)
            result=pd.DataFrame(columns=['ID', 'resname', 'resid', 
                                         'x_1', 'y_1', 'z_1', 
                                         'x_2', 'y_2', 'z_2',
                                         'x_3', 'y_3', 'z_3', 
                                         'x_4', 'y_4', 'z_4', 
                                         'x_5', 'y_5', 'z_5'], 
                                         data=[[target_id, x, j+1] + [0.0]*15 for j, x in enumerate(seq)])
            results[target_id] = {'df': result, 'best_score': float('inf')}  # High score for failed predictions
            
        if truncated:
            missing_len = len(original_seq) - len(seq)
            if missing_len > 0:
                zero_rows = pd.DataFrame([[target_id, original_seq[j], j+1] + [0.0]*15
                                          for j in range(len(seq), len(original_seq))],
                                         columns=result.columns)
                result = pd.concat([result, zero_rows], ignore_index=True)
                
        result['ID']=result.apply(lambda x: x.ID + '_' + str(x.resid), axis=1)
        result.to_csv('submission.csv', index=False, mode='a', header=(i==0))
        target_id=test_df.target_id[i]
        if len(seq)<=TRRNA_THRESHOLD:
            trRNA_input_ids.append(target_id)
            print('Send to trRNA', target_id)
        # results[target_id]=result  # Modified above to store dict with score
        torch.cuda.empty_cache()

    display(pd.read_csv('submission.csv'))

In [ ]:
import pdb
import os
import tempfile
import shutil
import subprocess

import tempfile

import glob

import string

import json
import os
import sys
import numpy as np
import torch
import torch.nn as nn

from collections import defaultdict
from argparse import ArgumentParser
from pathlib import Path

In [ ]:
import warnings
warnings.simplefilter("ignore", category=FutureWarning)

In [ ]:
torch.cuda.is_available(),torch.cuda.device_count()

In [ ]:
import os
import sys
import shutil
from pathlib import Path

input_base = '/kaggle/input'
work_base = '/kaggle/working'

required_dirs = [
    'folding',
    'network',
    'spot-rna',
    'model-1'
]

raise RuntimeError('Archived workspace deletion is disabled; use an isolated fresh run directory.')  # 慎用！确保你知道自己在做什么
os.makedirs(work_base, exist_ok=True)

for dir_name in required_dirs:
    src = os.path.join(input_base, dir_name)
    dst = os.path.join(work_base, dir_name)
    
    if os.path.exists(src):
        shutil.copytree(src, dst)
        print(f"Copied: {src} => {dst}")
    else:
        raise FileNotFoundError(f"关键目录缺失: {src}")
    
sys.path.append("/kaggle/working/")

In [ ]:
torch.cuda.is_available(),torch.cuda.device_count()

In [ ]:
import os
import numpy as np
import string


def parse_a3m(filename, limit=20000, rm_query_gap=True):
    seqs = []
    table = str.maketrans(dict.fromkeys(string.ascii_lowercase))

    # read file line by line
    n = 0
    for line in open(filename, "r"):
        if line[0] != '>' and len(line.strip()) > 0:
            seqs.append(
                line.rstrip().replace('W', 'A').replace('R', 'A').replace('Y', 'C').replace('E', 'A').replace('I',
                                                                                                              'A').replace(
                    'P', 'G').replace('T', 'U').translate(table))
            n += 1
            if n == limit:
                break

    # convert letters into numbers
    alphabet = np.array(list("AUCG-"), dtype='|S1').view(np.uint8)
    msa = np.array([list(s) for s in seqs], dtype='|S1').view(np.uint8)
    for i in range(alphabet.shape[0]):
        msa[msa == alphabet[i]] = i

        # treat all unknown characters as gaps
    msa[msa > 4] = 4
    if rm_query_gap:
        return msa[:, msa[0] < 4]
    return msa


def ss2mat(ss_seq):
    ss_mat = np.zeros((len(ss_seq), len(ss_seq)))
    stack = []
    stack1 = []
    stack2 = []
    stack3 = []
    stack_alpha = {alpha: [] for alpha in string.ascii_lowercase}
    for i, s in enumerate(ss_seq):
        if s == '(':
            stack.append(i)
        elif s == ')':
            ss_mat[i, stack.pop()] = 1
        elif s == '[':
            stack1.append(i)
        elif s == ']':
            ss_mat[i, stack1.pop()] = 1
        elif s == '{':
            stack2.append(i)
        elif s == '}':
            ss_mat[i, stack2.pop()] = 1
        elif s == '<':
            stack3.append(i)
        elif s == '>':
            ss_mat[i, stack3.pop()] = 1
        elif s.isalpha() and s.isupper():
            stack_alpha[s.lower()].append(i)
        elif s.isalpha() and s.islower():
            ss_mat[i, stack_alpha[s].pop()] = 1
        elif s in ['.', ',', '_', ':', '-']:
            continue
        else:
            raise ValueError(f'unk not: {s}!')
    allstacks = stack + stack1 + stack2 + stack3
    for _, stack in stack_alpha.items():
        allstacks += stack
    if len(allstacks) > 0:
        raise ValueError('Provided dot-bracket notation is not completely matched!')

    ss_mat += ss_mat.T
    return ss_mat


def parse_ct(ct_file, length=None):
    seq_ct = ''
    if length is None:
        length = int(open(ct_file).readlines()[0].split()[0])
    mat = np.zeros((length, length))
    for line in open(ct_file):
        items = line.split()
        if len(items) >= 6 and items[0].isnumeric() and items[2].isnumeric() and items[3].isnumeric() and items[
            4].isnumeric():
            seq_ct += items[1]
            if int(items[4]) > 0:
                mat[int(items[4]) - 1, int(items[5]) - 1] = 1
                mat[int(items[5]) - 1, int(items[4]) - 1] = 1
    return mat

In [ ]:
pkg_dir = '/kaggle/working'
from network.RNAformer import DistPredictor
from network.config import n_bins, obj
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
import time
from datetime import datetime

def predict(model, msa, ss_, args, window=150, shift=50):
    start_time = time.time()
    
    if ss_.shape[0] != msa.shape[-1]:
        raise ValueError(f'ss length {ss_.shape[0]}, msa length {msa.shape[1]}!')
    with torch.no_grad():
        feat = torch.from_numpy(msa).to(device)
        ss_ = torch.from_numpy(ss_).to(device)
        L = msa.shape[-1]
        res_id = torch.arange(L, device=device).view(1, L)
        if L > 300:  # predict by crops for long RNA
            pred_dict = {
                'contact': torch.zeros((L, L), device=device),
                'distance': {k: torch.zeros((L, L, n_bins['2D']['distance']), device=device) for k in
                             obj['2D']['distance']},
            }

            count_1d = torch.zeros((L)).to(device)
            count_2d = torch.zeros((L, L)).to(device)
            
            grids = np.arange(0, L - window + shift, shift)
            ngrids = grids.shape[0]
            print("ngrid:     ", ngrids)
            print("grids:     ", grids)
            print("windows:   ", window)

            idx_pdb = torch.arange(L).long().view(1, L)
            for i in range(ngrids):
                for j in range(i, ngrids):
                    start_1 = grids[i]
                    end_1 = min(grids[i] + window, L)
                    start_2 = grids[j]
                    end_2 = min(grids[j] + window, L)
                    sel = np.zeros((L)).astype(np.bool_)
                    sel[start_1:end_1] = True
                    sel[start_2:end_2] = True

                    input_msa = feat[:, sel]
                    input_ss = ss_[sel][:, sel]
                    mask = torch.sum(input_msa == 4, dim=-1) < .7 * sel.sum()  # remove too gappy sequences

                    input_msa = input_msa[mask]
                    input_idx = idx_pdb[:, sel]
                    input_res_id = res_id[:, sel]

                    print("running crop: %d-%d/%d-%d" % (start_1, end_1, start_2, end_2), input_msa.shape)
                    pred_gemos = model(input_msa, input_ss, res_id=input_res_id.to(device),
                                       msa_cutoff=args.nrows)['geoms']
                    weight = 1
                    sub_idx = input_idx[0].cpu()
                    sub_idx_2d = np.ix_(sub_idx, sub_idx)
                    count_2d[sub_idx_2d] += weight
                    count_1d[sub_idx] += weight

                    for k in obj['2D']:
                        if k == 'contact':
                            pred_dict['contact'][sub_idx_2d] += weight * pred_gemos['contact']
                        else:
                            for a in obj['2D'][k]:
                                pred_dict[k][a][sub_idx_2d] += weight * pred_gemos[k][a]
                    
                    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    print(f"[{current_time}] Completed crop {i}-{j}, took {time.time() - start_time:.2f} seconds")
                    
            for k in obj['2D']:
                if k == 'contact':
                    pred_dict['contact'] /= count_2d
                else:
                    for a in obj['2D'][k]:
                        if pred_dict[k][a].size().__len__() == 3:
                            pred_dict[k][a] /= count_2d[:, :, None]
                        else:
                            pred_dict[k][a] /= count_2d
        else:
            pred_dict = model(feat, ss_, res_id=res_id.to(device), msa_cutoff=args.nrows)['geoms']
            current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"[{current_time}] Completed prediction, took {time.time() - start_time:.2f} seconds")

    for l in pred_dict:
        if isinstance(pred_dict[l], dict):
            for k in pred_dict[l]:
                pred_dict[l][k] = pred_dict[l][k].cpu().detach().numpy()
        else:
            pred_dict[l] = pred_dict[l].cpu().detach().numpy()

    return pred_dict


In [ ]:
def main(args):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)
    torch.set_num_threads(args.cpu)
    device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    py = sys.executable

    out_dir = os.path.dirname(os.path.abspath(args.npz))
    os.makedirs(out_dir, exist_ok=True)

    cwd = os.getcwd()
    msa = parse_a3m(args.msa, limit=20000)

    # Check if ss_file is not provided
    if args.ss_file is None:
        # Predict SS by SPOT-RNA
        print('predict SS by SPOT-RNA')

        # Create a temporary directory
        tmp_dir = tempfile.TemporaryDirectory(prefix=out_dir + '/')
        spot_out_dir = tmp_dir.name

        # Check if the seq.fasta file doesn't exist, and create it by extracting the first 2 lines from the MSA file
        if not os.path.isfile(f'{spot_out_dir}/seq.fasta'):
            with open(f'{spot_out_dir}/seq.fasta', 'w') as fasta_file:
                with open(args.msa, 'r') as msa_file:
                    # Write the first 2 lines of MSA file to seq.fasta
                    for _ in range(2):
                        line = msa_file.readline()
                        fasta_file.write(line)

        # Modify Python environment path for SPOT-RNA
        # spot_py = py.replace('trRNA', 'spot_venv')
        spot_py = py

        # Change current working directory to SPOT_RNA directory
        os.chdir(f'{pkg_dir}/spot-rna')

        # If 'utils' directory exists, rename it to 'utils_spot'
        if os.path.isdir(f'utils'):
            shutil.move(f'utils', f'utils_spot')

        # Modify SPOT-RNA.py to use the new utils_spot directory
        with open('SPOT-RNA.py', 'r') as spot_script_file:
            spot_script = spot_script_file.read()


        # Run SPOT-RNA using subprocess with nohup equivalent in Python
        # print(spot_py)
        # print(spot_out_dir)
        with open(f'{out_dir}/spot.log', 'w') as log_file:
            # pdb.set_trace()
            subprocess.run(
                [spot_py, 'SPOT-RNA.py', '--inputs', f'{spot_out_dir}/seq.fasta', '--outputs', spot_out_dir, '--gpu', str(args.gpu)],
                stdout=log_file, stderr=subprocess.STDOUT
            )

        # Return to the original working directory
        os.chdir(cwd)

        prob_files = glob.glob(f'{spot_out_dir}/*.prob')
        if len(prob_files) == 0: raise ValueError(
            f'Fails to predict SS! Please refer to {out_dir}/spot.log to see what happened.')
        ss = np.loadtxt(prob_files[0])
        if (np.tril(ss) == 0).all() or (np.triu(ss) == 0).all():
            ss += ss.T
    else:
        if args.ss_fmt == 'dot_bracket':
            ss = ss2mat(open(args.ss_file).read().rstrip().splitlines()[-1].strip())
        elif args.ss_fmt == 'ct':
            ss = parse_ct(args.ss_file, length=len(msa[0]))
        elif args.ss_fmt == 'spot_prob':
            ss = np.loadtxt(args.ss_file)
            ss += ss.T
        if len(ss) != len(msa[0]):
            raise ValueError(f'The SS shape {ss.shape} mismatches the MSA shape {msa.shape}!')

    print('predict geometries')
    config = json.load(open(f'{args.model_pth}/config/model_1.json', 'r'))

    model = DistPredictor(dim_2d=config['channels'], layers_2d=config['n_blocks'])

    model_ckpt = torch.load(f'{args.model_pth}/models/model_1.pth.tar', map_location=device)
    model.load_state_dict(model_ckpt)
    model.eval()
    model.to(device)

    pred = predict(model, msa, ss, args=args)

    print('done!')
    print('saving......')
    np.savez_compressed(args.npz, **pred)


In [ ]:
from folding.utils_cst import npz2cst
from folding.utils_ros import fold_from_cst

In [ ]:
def fold(args):
    os.makedirs(os.path.dirname(os.path.abspath(args.OUT)), exist_ok=True)

    tmpdir = tempfile.TemporaryDirectory(prefix=args.TMPDIR + '/')
    args.tmpdir = tmpdir.name
    print('temp folder:     ', tmpdir.name)

    # parse npz into rosetta-format restraint files
    npz2cst(args)

    original_out = args.OUT
    base, ext = os.path.splitext(original_out)
    
    # perform energy minimization for each model
    args.OUT = f"{base}_model_1{ext}"
    fold_from_cst(args)
    for i in range(2,args.nmodels+1):
        os.system(f"cp {tmpdir.name}/model_{i}.pdb {base}_model_{i}{ext}");
    
    args.OUT = original_out

In [ ]:
import os
import pandas as pd
from biopandas.pdb import PandasPdb

def get_c1_coords(base="/kaggle/working/output_model_model", ext=".pdb", nmodels=5):
    c1_coords_all_models = {}
    scores = []
    for i in range(1, nmodels+1):
        model_path = f"{base}_{i}{ext}"
        ppdb = PandasPdb().read_pdb(model_path)
        atom_df = ppdb.df['ATOM']
        
        # Score the model
        score = score_pdb(model_path)
        scores.append(score)
        
        for resid, group in atom_df.groupby('residue_number'):
            resname = group['residue_name'].iloc[0]
            c1_coords = group[group['atom_name'] == "C1'"][['x_coord', 'y_coord', 'z_coord']].values
            
            if resid not in c1_coords_all_models:
                c1_coords_all_models[resid] = {'resname': resname, 'coords': []}
                
            if len(c1_coords) > 0:
                c1_coords_all_models[resid]['coords'].extend(c1_coords[0])
            else:
                c1_coords_all_models[resid]['coords'].extend([0.0, 0.0, 0.0])

    rna_id = os.path.basename(base).split('_')[1]
        
    result=pd.DataFrame(columns=['ID', 'resname', 'resid'] + 
    [item for sublist in [[f'x_{j+1}', f'y_{j+1}', f'z_{j+1}'] for j in range(int(len(c1_coords_all_models[1]['coords'])/3))] for item in sublist], 
                        data=[[f"{rna_id}_{resid}", data['resname'], resid] + 
                              [data['coords'][j] for j in range(len(data['coords']))] for resid, data in c1_coords_all_models.items()])
    return result, min(scores)  # Return the best (lowest) score


In [ ]:
ori_dir='/kaggle/input/stanford-rna-3d-folding/MSA'
target_dir='/kaggle/working/simplified-msa/MSA1'
os.makedirs(target_dir, exist_ok=True)
for filename in os.listdir(ori_dir):
    basename=filename.split('.')[0]
    if(basename not in trRNA_input_ids):
        continue
    if filename.endswith('.fasta'):
        with open(os.path.join(ori_dir, filename), 'r') as f:
            lines = f.readlines()
            with open(os.path.join(target_dir, filename), 'w') as wf:
                wf.writelines(lines[:2])

In [ ]:
class Main_Args:
    msa="/kaggle/input/simplified-msa/MSA1/R1107.MSA.fasta"
    npz = "/kaggle/working/R1107.npz"
    ss_file = None
    ss_fmt = "dot_bracket"
    gpu = "0"
    cpu = NUM_CPU
    model_pth="/kaggle/working/model-1"
    nrows=1000

In [ ]:
class Fold_Args:
    NPZ = "/kaggle/working/R1107.npz"
    FASTA = "/kaggle/input/simplified-msa/MSA1/R1107.MSA.fasta"
    OUT = "/kaggle/working/output_model.pdb"
    
    TMPDIR = "/kaggle/working"
    nmodels = 5
    dcut = 0.45
    CPU = NUM_CPU

In [ ]:
main_args = Main_Args()
fold_args = Fold_Args()

for i, input_id in enumerate(trRNA_input_ids):
    main_args.npz = f'/kaggle/working/{input_id}.npz'
    main_args.msa = f'/kaggle/working/simplified-msa/MSA1/{input_id}.MSA.fasta'
    fold_args.NPZ = f'/kaggle/working/{input_id}.npz'
    fold_args.FASTA = f'/kaggle/working/simplified-msa/MSA1/{input_id}.MSA.fasta'
    fold_args.OUT = f'/kaggle/working/output_{input_id}.pdb'
    print('main_args', main_args)
    print('fold_args', fold_args)
    main(main_args)
    fold(fold_args)
    result, rnaformer_score = get_c1_coords(base=f'/kaggle/working/output_{input_id}_model', ext='.pdb', nmodels=fold_args.nmodels)
        
    result_new = results[input_id]['df']
    
    result_new[['x_5', 'y_5', 'z_5']] = result[['x_1', 'y_1', 'z_1']].values
    print(f"Inserted RNAformer result into Protenix model 5 for {input_id}")
    
    results[input_id] = result_new  # Update results with the DataFrame only
    torch.cuda.empty_cache()
    all_results = pd.concat([r['df'] if isinstance(r, dict) else r for r in results.values()], ignore_index=True)
    all_results.to_csv('submission.csv', index=False, header=True)
    

In [ ]:
all_results = pd.concat(results, ignore_index=True)
all_results.to_csv('submission.csv', index=False, header=True)

In [ ]:
display(pd.read_csv('submission.csv'))